# Practice: Is an Off-the-Shelf Tagger Good Enough for Your Corpus?

**Computational Text Analysis Pack · Module 16 — Sequence Labelling**

*Technique:* Evaluating a tagger on your own material before depending on it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/06-supervised-classification/16c_practice_tagging_project.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **You want to count adjectives in a nineteenth-century corpus. The tagger was
> trained on modern newswire. Is it good enough, and how would you find out
> without annotating thousands of tokens?**

The answer is a small, careful annotation exercise — which is also the cheapest
useful thing most DH projects never do.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import nltk, numpy as np
sents = []
for passage in df_colonial["text"].head(300).astype(str):
    for raw in passage.split("."):
        toks = tokenise(raw)
        if 6 <= len(toks) <= 24:
            sents.append(toks)
print(f"{len(sents)} candidate sentences from pre-1929 prose")

rng = np.random.default_rng(1)
sample = [sents[i] for i in rng.choice(len(sents), size=25, replace=False)]
tagged = [nltk.pos_tag(s) for s in sample]
print("\nFirst two, as the tagger sees them:\n")
for s in tagged[:2]:
    print("   " + " ".join(f"{w}/{t}" for w, t in s))

In [ ]:
# Where does a modern tagger go wrong on older prose? Look for the signals.
SUSPECT = []
for s in tagged:
    for w, t in s:
        # Archaic verb forms and second-person singular are the classic failures.
        if w.endswith(("eth", "est")) or w in {"thou","thee","thy","hath","doth","art"}:
            SUSPECT.append((w, t))
print(f"archaic forms found: {len(SUSPECT)}")
for w, t in SUSPECT[:12]:
    print(f"   {w:<12}tagged {t}")
print()
counts = Counter(t for s in tagged for _, t in s)
print("tag distribution on this sample:")
for t, n in counts.most_common(10):
    print(f"   {t:<8}{n:>5}")

In [ ]:
# The measurement that matters: hand-check ONE tag you care about.
TAG_OF_INTEREST = "JJ"       # adjectives
candidates = [(w, t) for s in tagged for w, t in s if t == TAG_OF_INTEREST]
print(f"the tagger calls {len(candidates)} tokens {TAG_OF_INTEREST} in this sample.\n")
print("Check these by eye — are they all adjectives?\n")
for w, t in candidates[:20]:
    print(f"   {w}")
print()
print("Then estimate precision, and do the harder half: scan the sentences for")
print("adjectives it MISSED, which gives you recall. Twenty minutes of work")
print("gives you a number you can put in a footnote.")

In [ ]:
# Suppose your check gave these. Replace with your own.
CHECKED, CORRECT, MISSED = len(candidates), int(len(candidates)*0.82), 6
prec = CORRECT / max(CHECKED, 1)
rec = CORRECT / max(CORRECT + MISSED, 1)
f1 = 2*prec*rec/max(prec+rec, 1e-9)
print(f"precision {prec:.2f}   recall {rec:.2f}   F1 {f1:.2f}")
print()
total_jj = 0
for s in sents:
    total_jj += sum(1 for _, t in nltk.pos_tag(s) if t == TAG_OF_INTEREST)
print(f"raw count across the corpus sample : {total_jj:,} adjectives")
print(f"precision-adjusted estimate        : {total_jj*prec:,.0f}")
print(f"recall-adjusted estimate           : {total_jj*prec/max(rec,1e-9):,.0f}")
print()
print("Three numbers, one dataset. Which you report changes the finding, and")
print("only the first is available if you never checked.")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(["raw", "precision-adj", "recall-adj"],
       [total_jj, total_jj*prec, total_jj*prec/max(rec,1e-9)], color="steelblue")
ax.set_ylabel("estimated adjectives"); ax.set_title("The same corpus, three counts")
plt.tight_layout()

## What this licenses

**Defensible:** "Part-of-speech tags were assigned with [tagger, version]. On a
hand-checked sample of 25 sentences, adjective precision was [X] and recall [Y];
counts below are precision-adjusted."

**Not defensible:** a raw tag count from a tagger trained on a different period,
register or language, with no error estimate. It is not that the number is
wrong — it is that nobody, including you, knows how wrong.

**And a limit worth stating:** this tagger has no model of your period at all.
For heavily archaic material, module 14c's normalisation before tagging will do
more for accuracy than any amount of adjusting afterwards.

### 🔧 Your turn 1

Run the same audit on a different tag — nouns, or verbs. Is the error rate the same across tags?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

It is usually not. Taggers are near-perfect on determiners and prepositions —
closed classes with fixed membership — and worst on the open classes where new
and archaic vocabulary lives, which are exactly the ones a literary study cares
about. Reporting one overall tagger accuracy therefore tells a reader almost
nothing about the tag you actually counted, and the per-tag figure is the one
worth the twenty minutes.

</details>